In [1]:
import torch
import math
from torch_geometric.data import Data


x0_input = [0, 0, 0, 0, 0, 0, 0]
z0_input = [0, 10, 20, 30, 40, 50, 60]


def build_mooring_graph(
    x0_input,
    z0_input,
    line_length: float,
    ea: float = 1.0e7,
    mass_per_length: float = 100.0,
    diameter: float = 0.1,
    area: float = 0.01,
    submerged_weight_per_length: float = 0.0,
    z_bed: float = 0.0,
    k_b: float = 0.0,
    c_b: float = 0.0,
):

    # Convert the input coordinates to PyTorch tensors.
    # These are the node coordinates of the initial mooring-line geometry
    x0 = torch.tensor(x0_input, dtype=torch.float32)
    z0 = torch.tensor(z0_input, dtype=torch.float32)
    

    # Number of nodes is now inferred directly from the geometry input.
    num_nodes = len(x0)

    # Basic consistency checks.
    if num_nodes < 2:
        raise ValueError("At least 2 nodes are required.")

    if len(z0) != num_nodes:
        raise ValueError("x0_input and z0_input must have the same length.")

    if line_length <= 0.0:
        raise ValueError("line_length must be positive.")

    # ------------------------------------------------------------------
    # 2. REFERENCE ARC-LENGTH COORDINATE s
    # ------------------------------------------------------------------

        # Reference arc-length coordinate along the line.
    # Since the geometry is now prescribed by the user, we still define
    # a normalized coordinate from 0 to 1 along the node ordering.
    s = torch.linspace(0.0, line_length, num_nodes)
    s_over_L = s / line_length

    # ------------------------------------------------------------------
    # 3. INITIAL GEOMETRY OF THE LINE
    # ------------------------------------------------------------------

    # pos stores the 2D coordinates of the prescribed initial geometry.
    pos = torch.stack([x0, z0], dim=1)

    # ------------------------------------------------------------------
    # 4. NODE TYPE FLAGS
    # ------------------------------------------------------------------
    # Node-type flags
    is_anchor = torch.zeros(num_nodes, dtype=torch.float32)
    is_fairlead = torch.zeros(num_nodes, dtype=torch.float32)
    is_internal = torch.ones(num_nodes, dtype=torch.float32)

    is_anchor[0] = 1.0
    is_fairlead[-1] = 1.0
    is_internal[0] = 0.0
    is_internal[-1] = 0.0

    # Explicit node type id:
    # 0 = anchor
    # 1 = internal
    # 2 = fairlead
    node_type_id = torch.ones(num_nodes, dtype=torch.float32)
    node_type_id[0] = 0.0
    node_type_id[-1] = 2.0


    # --------------------------------------------------------------
    # STATIC PHYSICAL NODE PROPERTIES
    # --------------------------------------------------------------

    # Uniform segment length based on the reference line length.
    segment_length_nominal = line_length / (num_nodes - 1)

    # Nodal lumped mass:
    # a simple first approximation is mass_per_length times nominal segment length.
    # End nodes get half-contribution, interior nodes get full contribution.
    nodal_mass = torch.full(
        (num_nodes,),
        mass_per_length * segment_length_nominal,
        dtype=torch.float32
    )
    nodal_mass[0] *= 0.5
    nodal_mass[-1] *= 0.5

    # Diameter and area stored per node as static physical attributes.
    diameter_node = torch.full((num_nodes,), diameter, dtype=torch.float32)
    area_node = torch.full((num_nodes,), area, dtype=torch.float32)

    # Submerged weight contribution per node.
    submerged_weight_node = torch.full(
        (num_nodes,),
        submerged_weight_per_length * segment_length_nominal,
        dtype=torch.float32
    )
    submerged_weight_node[0] *= 0.5
    submerged_weight_node[-1] *= 0.5

    # Seabed level stored per node.
    # For now this assumes a flat seabed, same z_bed for all nodes.
    z_bed_node = torch.full((num_nodes,), z_bed, dtype=torch.float32)

    # Flag indicating whether a node can potentially interact with seabed.
    # For a mooring line, all nodes may potentially touch the seabed.
    can_touch_seabed = torch.ones(num_nodes, dtype=torch.float32)
   
    # ------------------------------------------------------------------
    # 5. STATIC NODE FEATURES
    # ------------------------------------------------------------------

    # We now assemble richer static node features.
    #
    # Node feature columns:
    # 0) s_over_L               -> normalized position along the line
    # 1) is_anchor              -> anchor flag
    # 2) is_fairlead            -> fairlead flag
    # 3) is_internal            -> internal-node flag
    # 4) node_type_id           -> 0=anchor, 1=internal, 2=fairlead
    # 5) x0                     -> reference x-coordinate
    # 6) z0                     -> reference z-coordinate
    # 7) nodal_mass             -> lumped nodal mass
    # 8) diameter_node          -> local line diameter
    # 9) area_node              -> local cross-sectional area
    # 10) submerged_weight_node -> submerged weight contribution at node
    # 11) z_bed_node            -> flat seabed elevation
    # 12) can_touch_seabed      -> node touching the seabed
    x = torch.stack(
    [
        s_over_L,                # 0
        is_anchor,               # 1
        is_fairlead,             # 2
        is_internal,             # 3
        node_type_id,            # 4
        x0,                      # 5
        z0,                      # 6
        nodal_mass,              # 7
        diameter_node,           # 8
        area_node,               # 9
        submerged_weight_node,   # 10
        z_bed_node,              # 11
        can_touch_seabed,        # 12
    ],
    dim=1
)

    # ------------------------------------------------------------------
    # 6. GRAPH CONNECTIVITY (edge_index)
    # ------------------------------------------------------------------

    # For a single mooring line, the natural graph is a chain:
    #
    # node 0 -- node 1 -- node 2 -- ... -- node N-1
    #
    # In PyTorch Geometric, edge_index has shape [2, num_edges].
    # Each column is one directed edge: [source_node, target_node].

    senders = []
    receivers = []

    # Loop through all adjacent node pairs.
    for i in range(num_nodes - 1):
        # Forward edge: node i sends information to node i+1.
        senders.append(i)
        receivers.append(i + 1)

        # Backward edge: node i+1 sends information to node i.
        # We include both directions because message passing in GNNs
        # is usually easier when the graph is explicitly bidirectional.
        senders.append(i + 1)
        receivers.append(i)

    # Convert Python lists to a PyTorch tensor of shape [2, num_edges].
    edge_index = torch.tensor([senders, receivers], dtype=torch.long)

    # ------------------------------------------------------------------
    # 7. EDGE FEATURES
    # ------------------------------------------------------------------

    # Richer edge features based on neighboring reference-node coordinates.
    edge_features = []

    for i in range(num_nodes - 1):
        dx = float(x0[i + 1] - x0[i])
        dz = float(z0[i + 1] - z0[i])

        # Physical unstretched segment length from the prescribed reference geometry.
        segment_length = math.sqrt(dx**2 + dz**2)
        segment_length_over_L = segment_length / line_length

        # Unit tangent vector of the segment in reference geometry.
        if segment_length > 0.0:
            tangent_x = dx / segment_length
            tangent_z = dz / segment_length
        else:
            tangent_x = 0.0
            tangent_z = 0.0

        # Explicit edge type id:
        # 0 = standard mooring-segment edge
        edge_type_id = 0.0

        forward_features = [
            segment_length,              # 0: segment length [m]
            segment_length_over_L,       # 1: normalized segment length [-]
            ea,                          # 2: axial stiffness EA [N]
            mass_per_length,             # 3: mass per unit length [kg/m]
            diameter,                    # 4: line diameter [m]
            tangent_x,                   # 5: tangent x-component
            tangent_z,                   # 6: tangent z-component
            edge_type_id,                # 7: edge type id
        ]

        backward_features = [
            segment_length,
            segment_length_over_L,
            ea,
            mass_per_length,
            diameter,
            tangent_x,
            tangent_z,
            edge_type_id,
        ]

        edge_features.append(forward_features)
        edge_features.append(backward_features)

    edge_attr = torch.tensor(edge_features, dtype=torch.float32)
    # ------------------------------------------------------------------
    # 8. CREATE THE GRAPH OBJECT
    # ------------------------------------------------------------------

    # PyTorch Geometric Data object stores all graph information.
    data = Data(
        x=x,                  # static node feature matrix
        edge_index=edge_index,  # graph connectivity
        edge_attr=edge_attr,    # static edge feature matrix
        pos=pos               # node coordinates for visualization or geometry-based models
    )

    # ------------------------------------------------------------------
    # 9. STORE EXTRA METADATA
    # ------------------------------------------------------------------

    data.num_nodes_total = num_nodes
    data.line_length = line_length
    data.ea = ea
    data.mass_per_length = mass_per_length
    data.diameter = diameter
    data.area = area
    data.submerged_weight_per_length = submerged_weight_per_length

    # Seabed-related metadata
    data.z_bed = z_bed
    data.k_b = k_b
    data.c_b = c_b
    data.z_bed_node = z_bed_node

    return data


def print_graph_summary(data: Data):
    """
    Print a readable summary of the graph.
    """

    print("----- GRAPH SUMMARY -----")
    print(f"Number of nodes: {data.num_nodes_total}")
    print(f"Line length [m]: {data.line_length}")
    print(f"Axial stiffness EA [N]: {data.ea}")
    print(f"Mass per unit length [kg/m]: {data.mass_per_length}")
    print()

    print("Node feature matrix x shape:", data.x.shape)
    print("Edge index shape:", data.edge_index.shape)
    print("Edge feature matrix shape:", data.edge_attr.shape)
    print("Node position matrix pos shape:", data.pos.shape)
    print()

    print("Node feature columns:")
    print("0  -> s_over_L")
    print("1  -> is_anchor")
    print("2  -> is_fairlead")
    print("3  -> is_internal")
    print("4  -> node_type_id")
    print("5  -> x0")
    print("6  -> z0")
    print("7  -> nodal_mass")
    print("8  -> diameter_node")
    print("9  -> area_node")
    print("10 -> submerged_weight_node")
    print("11 -> z_bed_node")
    print("12 -> can_touch_seabed")

    print("Edge feature columns:")
    print("0 -> segment_length")
    print("1 -> segment_length_over_L")
    print("2 -> EA")
    print("3 -> mass_per_length")
    print("4 -> diameter")
    print("5 -> tangent_x")
    print("6 -> tangent_z")
    print("7 -> edge_type_id")
    print()

    print("First 5 node features:")
    print(data.x[:5])
    print()

    print("First 10 edges:")
    print(data.edge_index[:, :10])
    print()

    print("First 10 edge features:")
    print(data.edge_attr[:10])
    print()

    print("First 5 node positions:")
    print(data.pos[:5])
    print()


if __name__ == "__main__":
    # Example usage:
    x0_input = [0, 0, 0, 0, 0, 0, 0]
    z0_input = [0, 10, 20, 30, 40, 50, 60]

    graph = build_mooring_graph(
        x0_input=x0_input,
        z0_input=z0_input,
        line_length=60.0,
        ea=2.0e7,
        mass_per_length=120.0,
        diameter=0.12,
        area=0.008,
        submerged_weight_per_length=450.0,
        z_bed=0.0,
        k_b=1.0e5,
        c_b=1.0e3,
    )

    # Print a summary so you can inspect the graph.
    print_graph_summary(graph)

----- GRAPH SUMMARY -----
Number of nodes: 7
Line length [m]: 60.0
Axial stiffness EA [N]: 20000000.0
Mass per unit length [kg/m]: 120.0

Node feature matrix x shape: torch.Size([7, 13])
Edge index shape: torch.Size([2, 12])
Edge feature matrix shape: torch.Size([12, 8])
Node position matrix pos shape: torch.Size([7, 2])

Node feature columns:
0  -> s_over_L
1  -> is_anchor
2  -> is_fairlead
3  -> is_internal
4  -> node_type_id
5  -> x0
6  -> z0
7  -> nodal_mass
8  -> diameter_node
9  -> area_node
10 -> submerged_weight_node
11 -> z_bed_node
12 -> can_touch_seabed
Edge feature columns:
0 -> segment_length
1 -> segment_length_over_L
2 -> EA
3 -> mass_per_length
4 -> diameter
5 -> tangent_x
6 -> tangent_z
7 -> edge_type_id

First 5 node features:
tensor([[0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 6.0000e+02, 1.2000e-01, 8.0000e-03, 2.2500e+03, 0.0000e+00,
         1.0000e+00],
        [1.6667e-01, 0.0000e+00, 0.0000e+00, 1.0000e+00, 1.00

In [2]:
from torch.utils.data import Dataset


class MooringSequenceDataset(Dataset):
    """
    Short-history-window dataset for a mooring-line GAT-LSTM.

    This dataset sits ON TOP OF the static graph you already built.

    The graph contains:
    - node connectivity
    - static node features
    - static edge features
    - initial/reference geometry

    This dataset adds the dynamic time-dependent features:
    - displacement
    - velocity
    - acceleration
    - tension

    Each training sample is:
    - input:  past history_len time steps
    - target: next future_len time steps

    Dynamic node features used here:
    --------------------------------
    [u_x, u_z, v_x, v_z, a_x, a_z, T]

    where:
    - u_x, u_z : displacement components at each node
    - v_x, v_z : velocity components at each node
    - a_x, a_z : acceleration components at each node
    - T        : tension at each node

    Notes
    -----
    If you later want absolute positions, you can reconstruct them as:
        x_abs = x_ref + u_x
        z_abs = z_ref + u_z
    using the node reference coordinates already stored in the graph.
    """

    def __init__(
        self,
        graph_data,
        u_x: torch.Tensor,
        u_z: torch.Tensor,
        v_x: torch.Tensor,
        v_z: torch.Tensor,
        a_x: torch.Tensor,
        a_z: torch.Tensor,
        tension: torch.Tensor,
        history_len: int,
        future_len: int = 1,
        target_mode: str = "displacement_and_tension",
        contact_tol: float = 1e-6,  
    ):
        
        """
        Parameters
        ----------
        graph_data : torch_geometric.data.Data
            Static graph object you already created.

        u_x, u_z : torch.Tensor
            Displacement components with shape [t, N].

        v_x, v_z : torch.Tensor
            Velocity components with shape [t, N].

        a_x, a_z : torch.Tensor
            Acceleration components with shape [t, N].

        tension : torch.Tensor
            Tension at each node with shape [t, N].

        history_len : int
            Number of past time steps given to the model.

        future_len : int
            Number of future time steps to predict.

        target_mode : str
            Defines what the target contains.
            Options:
            - "displacement_and_tension"
            - "absolute_position_and_tension"
        """

        self.graph_data = graph_data
        self.history_len = history_len
        self.future_len = future_len
        self.target_mode = target_mode

        # --------------------------------------------------------------
        # 1. BASIC SHAPE CHECKS
        # --------------------------------------------------------------

        # All dynamic tensors must have the same shape [t, N].
        expected_shape = u_x.shape

        tensors = {
            "u_z": u_z,
            "v_x": v_x,
            "v_z": v_z,
            "a_x": a_x,
            "a_z": a_z,
            "tension": tension,
        }

        for name, tensor in tensors.items():
            if tensor.shape != expected_shape:
                raise ValueError(
                    f"{name} has shape {tensor.shape}, but expected {expected_shape}."
                )

        # Time dimension t and node dimension N.
        self.num_steps, self.num_nodes = expected_shape

        # Check graph node count matches time-series node count.
        if self.graph_data.num_nodes_total != self.num_nodes:
            raise ValueError(
                f"Graph has {self.graph_data.num_nodes_total} nodes, "
                f"but dynamic data has {self.num_nodes} nodes."
            )

        # Must have enough time steps for at least one window.
        if self.num_steps < history_len + future_len:
            raise ValueError(
                "Not enough time steps for the chosen history_len and future_len."
            )

        # --------------------------------------------------------------
        # 2. STORE DYNAMIC INPUT FEATURES
        # --------------------------------------------------------------

        # --------------------------------------------------------------
        # 2b. SEABED INTERACTION FEATURES
        # --------------------------------------------------------------

        # Reference node coordinates from the graph.
        x_ref = self.graph_data.pos[:, 0]   # [N]
        z_ref = self.graph_data.pos[:, 1]   # [N]

        # Absolute position of each node over time:
        # x_abs = x_ref + u_x
        # z_abs = z_ref + u_z
        x_abs = u_x + x_ref.unsqueeze(0)    # [t, N]
        z_abs = u_z + z_ref.unsqueeze(0)    # [t, N]

        # Flat seabed level stored per node.
        z_bed_node = self.graph_data.z_bed_node.unsqueeze(0)   # [1, N]

        # Contact flag:
        # 1 if node is at or below seabed, 0 otherwise.
        contact_flag = (z_abs <= z_bed_node + contact_tol).float()

        # Penetration depth:
        # max(0, z_bed - z_abs)
        penetration_depth = torch.clamp(z_bed_node - z_abs, min=0.0)

        # Paper-style seabed vertical reaction magnitude:
        # b_bed = -k_b (z - z_bed) - c_b * z_dot, when z <= z_bed
        #
        # Since penetration_depth = (z_bed - z_abs) when in contact,
        # this becomes:
        # b_bed = k_b * penetration_depth - c_b * v_z
        #
        # We set it to zero when out of contact.
        bed_reaction_z = self.graph_data.k_b * penetration_depth - self.graph_data.c_b * v_z
        bed_reaction_z = torch.where(contact_flag > 0.0, bed_reaction_z, torch.zeros_like(bed_reaction_z))

        # Optional clamp: seabed cannot pull downward, only push upward.
        bed_reaction_z = torch.clamp(bed_reaction_z, min=0.0)

        # Stack dynamic node features into a single tensor of shape [t, N, F_dynamic].
        #
        # Feature order:
        # 0 -> u_x
        # 1 -> u_z
        # 2 -> v_x
        # 3 -> v_z
        # 4 -> a_x
        # 5 -> a_z
        # 6 -> tension
        self.dynamic_features = torch.stack(
            [
                u_x,               # 0
                u_z,               # 1
                v_x,               # 2
                v_z,               # 3
                a_x,               # 4
                a_z,               # 5
                tension,           # 6
                contact_flag,      # 7
                penetration_depth, # 8
                bed_reaction_z,    # 9
            ],
            dim=-1
        )

        self.dynamic_feature_names = [
                "u_x",
                "u_z",
                "v_x",
                "v_z",
                "a_x",
                "a_z",
                "tension",
                "contact_flag",
                "penetration_depth",
                "bed_reaction_z",
            ]


        # --------------------------------------------------------------
        # 2c. DYNAMIC EDGE FEATURES FROM NODE SEABED STATE
        # --------------------------------------------------------------

        # edge_index has shape [2, E]
        # source_nodes and target_nodes each have shape [E]
        source_nodes = self.graph_data.edge_index[0]
        target_nodes = self.graph_data.edge_index[1]

        # Gather endpoint node quantities for every edge and every time step.
        #
        # If contact_flag has shape [t, N], then:
        # contact_src and contact_tgt have shape [t, E]
        contact_src = contact_flag[:, source_nodes]
        contact_tgt = contact_flag[:, target_nodes]

        penetration_src = penetration_depth[:, source_nodes]
        penetration_tgt = penetration_depth[:, target_nodes]

        bed_reaction_src = bed_reaction_z[:, source_nodes]
        bed_reaction_tgt = bed_reaction_z[:, target_nodes]

        # 1) Edge contact fraction
        # 0.0 -> both endpoints free
        # 0.5 -> one endpoint in contact, one free
        # 1.0 -> both endpoints in contact
        edge_contact_fraction = 0.5 * (contact_src + contact_tgt)

        # 2) Mean penetration depth over the segment endpoints
        edge_penetration_mean = 0.5 * (penetration_src + penetration_tgt)

        # 3) Mean seabed reaction over the segment endpoints
        edge_bed_reaction_mean = 0.5 * (bed_reaction_src + bed_reaction_tgt)

        # 4) Touchdown / lift-off transition flag
        # 1 if one endpoint is in contact and the other is not, else 0
        edge_touchdown_flag = (contact_src != contact_tgt).float()

        # Stack dynamic edge features into shape [t, E, F_edge_dynamic]
        self.dynamic_edge_features = torch.stack(
            [
                edge_contact_fraction,    # 0
                edge_penetration_mean,    # 1
                edge_bed_reaction_mean,   # 2
                edge_touchdown_flag,      # 3
            ],
            dim=-1
        )

        self.dynamic_edge_feature_names = [
            "edge_contact_fraction",
            "edge_penetration_mean",
            "edge_bed_reaction_mean",
            "edge_touchdown_flag",
        ]

        # --------------------------------------------------------------
        # 3. STORE TARGETS
        # --------------------------------------------------------------

        # Reference geometry coordinates come from the graph.
        # They are stored in the static node feature matrix or in graph.pos.
        x_ref = self.graph_data.pos[:, 0]   # shape [N]
        z_ref = self.graph_data.pos[:, 1]   # shape [N]

        if target_mode == "displacement_and_tension":
            # Target is future displacement and future tension.
            #
            # Target feature order:
            # 0 -> u_x
            # 1 -> u_z
            # 2 -> tension
            self.targets = torch.stack(
                [u_x, u_z, tension],
                dim=-1
            )
            self.target_feature_names = ["u_x", "u_z", "tension"]

        elif target_mode == "absolute_position_and_tension":
            # Convert displacement to absolute position using:
            # x_abs = x_ref + u_x
            # z_abs = z_ref + u_z
            #
            # Broadcasting works because:
            # x_ref and z_ref have shape [N]
            # u_x and u_z have shape [T, N]
            x_abs = u_x + x_ref.unsqueeze(0)
            z_abs = u_z + z_ref.unsqueeze(0)

            self.targets = torch.stack(
                [x_abs, z_abs, tension],
                dim=-1
            )
            self.target_feature_names = ["x_abs", "z_abs", "tension"]

        else:
            raise ValueError(
                "target_mode must be either "
                "'displacement_and_tension' or 'absolute_position_and_tension'."
            )

        # --------------------------------------------------------------
        # 4. NUMBER OF AVAILABLE WINDOWS
        # --------------------------------------------------------------

        # Example:
        # if t=100, history_len=10, future_len=2,
        # valid start indices are:
        # 0 ... 88
        #
        # because:
        # input  uses [start : start+10]
        # target uses [start+10 : start+12]
        self.num_windows = self.num_steps - self.history_len - self.future_len + 1

    def __len__(self):
        """
        Number of sliding windows available in the dataset.
        """
        return self.num_windows

    def __getitem__(self, idx):
        """
        Returns one training sample.

        Output dictionary:
        ------------------
        graph:
            The static graph object.

        x_seq:
            Input sequence of dynamic node features.
            Shape: [history_len, N, F_dynamic]

        y_seq:
            Target sequence.
            Shape: [future_len, N, F_target]

        start_idx:
            Starting time index of the sample.
        """
        if idx < 0 or idx >= self.num_windows:
            raise IndexError("Sample index out of range.")

        # Input window indices
        input_start = idx
        input_end = idx + self.history_len

        # Target window indices
        target_start = input_end
        target_end = target_start + self.future_len

        # Slice the dynamic feature tensor.
        x_seq = self.dynamic_features[input_start:input_end]   # [history_len, N, 10]
        edge_seq = self.dynamic_edge_features[input_start:input_end]   # [history_len, E, 4]

        # Slice the target tensor.
        y_seq = self.targets[target_start:target_end]          # [future_len, N, 3]

        return {
            "graph": self.graph_data,
            "x_seq": x_seq,
            "edge_seq": edge_seq,
            "y_seq": y_seq,
            "start_idx": idx,
        }


def print_dataset_summary(dataset: MooringSequenceDataset):
    """
    Print dataset information for inspection.
    """
    print("----- DATASET SUMMARY -----")
    print(f"Number of time steps: {dataset.num_steps}")
    print(f"Number of nodes: {dataset.num_nodes}")
    print(f"History length: {dataset.history_len}")
    print(f"Future length: {dataset.future_len}")
    print(f"Number of windows: {len(dataset)}")
    print()

    print("Dynamic input feature names:")
    print(dataset.dynamic_feature_names)
    print()

    print("Dynamic edge feature names:")
    print(dataset.dynamic_edge_feature_names)
    print()

    print("Target feature names:")
    print(dataset.target_feature_names)
    print()

    sample = dataset[0]

    print("First sample shapes:")
    print("x_seq shape:", tuple(sample["x_seq"].shape))
    print("edge_seq shape:", tuple(sample["edge_seq"].shape))
    print("y_seq shape:", tuple(sample["y_seq"].shape))
    print()

    print("Interpretation:")
    print("[history_len, N, 10] -> node-wise dynamic inputs")
    print("[history_len, E, 4]  -> edge-wise dynamic seabed inputs")
    print("[future_len, N, 3]   -> target outputs")


if __name__ == "__main__":
    # ------------------------------------------------------------------
    # Example initial geometry (7 nodes)
    # Replace these with the node coordinates you want to use.
    # ------------------------------------------------------------------
    x0_input = [0, 0, 0, 0, 0, 0, 0]
    z0_input = [0, 10, 20, 30, 40, 50, 60]

    # ------------------------------------------------------------------
    # Build graph
    # ------------------------------------------------------------------
    graph = build_mooring_graph(
        x0_input=x0_input,
        z0_input=z0_input,
        line_length=60.0,
        ea=2.0e7,
        mass_per_length=120.0,
        diameter=0.12,
        area=0.008,
        submerged_weight_per_length=450.0,
        z_bed=0.0,
        k_b=1.0e5,
        c_b=1.0e3,
    )

    # ------------------------------------------------------------------
    # EXAMPLE DYNAMIC DATA
    # ------------------------------------------------------------------

    # t = total number of time steps
    # N = total number of nodes from the graph
    t = 100
    N = graph.num_nodes_total

    # Fake dynamic data only for testing dimensions.
    # Replace all of these with your real simulation outputs later.
    u_x = torch.randn(t, N) * 0.10
    u_z = torch.randn(t, N) * 0.05

    v_x = torch.randn(t, N) * 0.20
    v_z = torch.randn(t, N) * 0.20

    a_x = torch.randn(t, N) * 0.50
    a_z = torch.randn(t, N) * 0.50

    tension = 1000.0 + torch.randn(t, N) * 50.0

    # ------------------------------------------------------------------
    # CREATE DATASET
    # ------------------------------------------------------------------

    dataset = MooringSequenceDataset(
        graph_data=graph,
        u_x=u_x,
        u_z=u_z,
        v_x=v_x,
        v_z=v_z,
        a_x=a_x,
        a_z=a_z,
        tension=tension,
        history_len=10,
        future_len=1,
        target_mode="displacement_and_tension",
        contact_tol=1e-6,
    )

    print_graph_summary(graph)
    print_dataset_summary(dataset)

    # Inspect one sample
    sample = dataset[0]

    print("\nFirst sample input sequence shape:")
    print(sample["x_seq"].shape)

    print("\nFirst sample edge sequence shape:")
    print(sample["edge_seq"].shape)

    print("\nFirst sample target sequence shape:")
    print(sample["y_seq"].shape)

    print("\nFirst sample input sequence:")
    print(sample["x_seq"])

    print("\nFirst sample target sequence:")
    print(sample["y_seq"])

----- GRAPH SUMMARY -----
Number of nodes: 7
Line length [m]: 60.0
Axial stiffness EA [N]: 20000000.0
Mass per unit length [kg/m]: 120.0

Node feature matrix x shape: torch.Size([7, 13])
Edge index shape: torch.Size([2, 12])
Edge feature matrix shape: torch.Size([12, 8])
Node position matrix pos shape: torch.Size([7, 2])

Node feature columns:
0  -> s_over_L
1  -> is_anchor
2  -> is_fairlead
3  -> is_internal
4  -> node_type_id
5  -> x0
6  -> z0
7  -> nodal_mass
8  -> diameter_node
9  -> area_node
10 -> submerged_weight_node
11 -> z_bed_node
12 -> can_touch_seabed
Edge feature columns:
0 -> segment_length
1 -> segment_length_over_L
2 -> EA
3 -> mass_per_length
4 -> diameter
5 -> tangent_x
6 -> tangent_z
7 -> edge_type_id

First 5 node features:
tensor([[0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 6.0000e+02, 1.2000e-01, 8.0000e-03, 2.2500e+03, 0.0000e+00,
         1.0000e+00],
        [1.6667e-01, 0.0000e+00, 0.0000e+00, 1.0000e+00, 1.00

In [3]:
import torch
import torch.nn as nn
from torch_geometric.nn import GATv2Conv


class MooringGATEncoder(nn.Module):
    """
    Spatial graph encoder for one history time step.

    It takes:
      - static node features from graph.x
      - static edge features from graph.edge_attr
      - dynamic node features for one time step x_t
      - dynamic edge features for one time step edge_t

    and returns:
      - node embeddings h_t for that time step

    Expected shapes for one sample:
      graph.x         : [N, n_static_node_features]
      graph.edge_index: [2, E]
      graph.edge_attr : [E, n_static_edge_features]
      x_t             : [N, n_dynamic_node_features]
      edge_t          : [E, n_dynamic_edge_features]

    Output:
      h_t             : [N, gat_hidden_dim]
    """

    def __init__(
        self,
        n_static_node_features: int = 13,
        n_dynamic_node_features: int = 10,
        n_static_edge_features: int = 8,
        n_dynamic_edge_features: int = 4,
        gat_hidden_dim: int = 64,
        gat_out_dim: int = 64,
        num_heads: int = 4,
        dropout: float = 0.1,
        use_layernorm: bool = True,
        add_residual_projection: bool = True,
    ):
        super().__init__()

        self.n_static_node_features = n_static_node_features
        self.n_dynamic_node_features = n_dynamic_node_features
        self.n_static_edge_features = n_static_edge_features
        self.n_dynamic_edge_features = n_dynamic_edge_features

        self.node_input_dim = n_static_node_features + n_dynamic_node_features
        self.edge_input_dim = n_static_edge_features + n_dynamic_edge_features

        self.gat_hidden_dim = gat_hidden_dim
        self.gat_out_dim = gat_out_dim
        self.num_heads = num_heads
        self.dropout = dropout
        self.use_layernorm = use_layernorm

        # -------------------------------------------------------------
        # GAT layer 1
        # concat=True => output dim = gat_hidden_dim * num_heads
        # -------------------------------------------------------------
        self.gat1 = GATv2Conv(
            in_channels=self.node_input_dim,
            out_channels=gat_hidden_dim,
            heads=num_heads,
            concat=True,
            dropout=dropout,
            edge_dim=self.edge_input_dim,
            add_self_loops=False,
            bias=True,
        )

        self.norm1 = nn.LayerNorm(gat_hidden_dim * num_heads) if use_layernorm else nn.Identity()

        # -------------------------------------------------------------
        # GAT layer 2
        # concat=False => output dim = gat_out_dim
        # -------------------------------------------------------------
        self.gat2 = GATv2Conv(
            in_channels=gat_hidden_dim * num_heads,
            out_channels=gat_out_dim,
            heads=1,
            concat=False,
            dropout=dropout,
            edge_dim=self.edge_input_dim,
            add_self_loops=False,
            bias=True,
        )

        self.norm2 = nn.LayerNorm(gat_out_dim) if use_layernorm else nn.Identity()

        self.act = nn.ELU()
        self.dropout_layer = nn.Dropout(dropout)

        # Optional residual projection from raw concatenated node input
        if add_residual_projection:
            self.residual_proj = nn.Linear(self.node_input_dim, gat_out_dim)
        else:
            self.residual_proj = None

    def forward(self, graph, x_t: torch.Tensor, edge_t: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        graph : torch_geometric.data.Data
            Static graph object containing graph.x, graph.edge_index, graph.edge_attr.
        x_t : torch.Tensor
            Dynamic node features at one time step, shape [N, n_dynamic_node_features].
        edge_t : torch.Tensor
            Dynamic edge features at one time step, shape [E, n_dynamic_edge_features].

        Returns
        -------
        h_t : torch.Tensor
            Encoded node embeddings for this time step, shape [N, gat_out_dim].
        """

        # -------------------------------------------------------------
        # Basic shape checks
        # -------------------------------------------------------------
        if x_t.dim() != 2:
            raise ValueError(f"x_t must have shape [N, F_dyn_node], got {tuple(x_t.shape)}")

        if edge_t.dim() != 2:
            raise ValueError(f"edge_t must have shape [E, F_dyn_edge], got {tuple(edge_t.shape)}")

        x_static = graph.x
        edge_index = graph.edge_index
        edge_static = graph.edge_attr

        if x_static.size(0) != x_t.size(0):
            raise ValueError(
                f"Node count mismatch: graph.x has {x_static.size(0)} nodes, "
                f"but x_t has {x_t.size(0)} nodes."
            )

        if edge_static.size(0) != edge_t.size(0):
            raise ValueError(
                f"Edge count mismatch: graph.edge_attr has {edge_static.size(0)} edges, "
                f"but edge_t has {edge_t.size(0)} edges."
            )

        if x_static.size(1) != self.n_static_node_features:
            raise ValueError(
                f"Expected {self.n_static_node_features} static node features, "
                f"got {x_static.size(1)}."
            )

        if x_t.size(1) != self.n_dynamic_node_features:
            raise ValueError(
                f"Expected {self.n_dynamic_node_features} dynamic node features, "
                f"got {x_t.size(1)}."
            )

        if edge_static.size(1) != self.n_static_edge_features:
            raise ValueError(
                f"Expected {self.n_static_edge_features} static edge features, "
                f"got {edge_static.size(1)}."
            )

        if edge_t.size(1) != self.n_dynamic_edge_features:
            raise ValueError(
                f"Expected {self.n_dynamic_edge_features} dynamic edge features, "
                f"got {edge_t.size(1)}."
            )

        # -------------------------------------------------------------
        # Concatenate static + dynamic features for this time step
        # -------------------------------------------------------------
        x_in = torch.cat([x_static, x_t], dim=-1)           # [N, 13 + 10] = [N, 23]
        edge_in = torch.cat([edge_static, edge_t], dim=-1) # [E, 8 + 4]  = [E, 12]

        # -------------------------------------------------------------
        # GAT block 1
        # -------------------------------------------------------------
        h = self.gat1(x_in, edge_index, edge_in)            # [N, gat_hidden_dim * num_heads]
        h = self.norm1(h)
        h = self.act(h)
        h = self.dropout_layer(h)

        # -------------------------------------------------------------
        # GAT block 2
        # -------------------------------------------------------------
        h = self.gat2(h, edge_index, edge_in)               # [N, gat_out_dim]
        h = self.norm2(h)

        # -------------------------------------------------------------
        # Optional residual connection from raw input
        # -------------------------------------------------------------
        if self.residual_proj is not None:
            h = h + self.residual_proj(x_in)

        h = self.act(h)
        h = self.dropout_layer(h)

        return h
    
# -------------------------------------------------------------
# Quick shape test on one dataset sample
# -------------------------------------------------------------
sample = dataset[0]

graph = sample["graph"] if isinstance(sample, dict) else sample[0]
x_seq = sample["x_seq"] if isinstance(sample, dict) else sample[1]
edge_seq = sample["edge_seq"] if isinstance(sample, dict) else sample[2]

print("graph.x shape      :", graph.x.shape)
print("graph.edge_attr    :", graph.edge_attr.shape)
print("x_seq shape        :", x_seq.shape)
print("edge_seq shape     :", edge_seq.shape)

encoder = MooringGATEncoder(
    n_static_node_features=13,
    n_dynamic_node_features=10,
    n_static_edge_features=8,
    n_dynamic_edge_features=4,
    gat_hidden_dim=64,
    gat_out_dim=64,
    num_heads=4,
    dropout=0.1,
)

x_t = x_seq[0]         # one history step: [N, 10]
edge_t = edge_seq[0]   # one history step: [E, 4]

with torch.no_grad():
    h_t = encoder(graph, x_t, edge_t)

print("x_t shape          :", x_t.shape)
print("edge_t shape       :", edge_t.shape)
print("h_t shape          :", h_t.shape)   # expected [N, 64]

graph.x shape      : torch.Size([7, 13])
graph.edge_attr    : torch.Size([12, 8])
x_seq shape        : torch.Size([10, 7, 10])
edge_seq shape     : torch.Size([10, 12, 4])
x_t shape          : torch.Size([7, 10])
edge_t shape       : torch.Size([12, 4])
h_t shape          : torch.Size([7, 64])


In [4]:
import torch
import torch.nn as nn


class NodeTemporalLSTM(nn.Module):
    """
    Temporal encoder that processes the sequence of spatial node embeddings
    produced by the MooringGATEncoder.

    Input:
        H : [history_len, N, gat_out_dim]

    Output:
        node_temporal : [N, lstm_hidden_dim]

    Interpretation:
        For each node i, we take its embedding sequence across time:
            H[:, i, :]  -> [history_len, gat_out_dim]
        and pass it through an LSTM.

    Notes:
        - This block is node-wise in time.
        - It does NOT mix nodes with each other.
        - Spatial coupling has already been handled by the GAT encoder.
    """

    def __init__(
        self,
        input_dim: int = 64,
        lstm_hidden_dim: int = 128,
        num_lstm_layers: int = 1,
        dropout: float = 0.1,
        bidirectional: bool = False,
        use_layernorm: bool = True,
        use_last_timestep: bool = True,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.lstm_hidden_dim = lstm_hidden_dim
        self.num_lstm_layers = num_lstm_layers
        self.bidirectional = bidirectional
        self.use_layernorm = use_layernorm
        self.use_last_timestep = use_last_timestep

        # PyTorch LSTM only uses dropout internally when num_layers > 1
        lstm_dropout = dropout if num_lstm_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,   # input will be [N, history_len, input_dim]
            dropout=lstm_dropout,
            bidirectional=bidirectional,
        )

        self.output_dim = lstm_hidden_dim * (2 if bidirectional else 1)

        self.norm = nn.LayerNorm(self.output_dim) if use_layernorm else nn.Identity()
        self.dropout_layer = nn.Dropout(dropout)

    def forward(self, H: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        H : torch.Tensor
            Sequence of node embeddings from the spatial encoder.
            Expected shape: [history_len, N, input_dim]

        Returns
        -------
        node_temporal : torch.Tensor
            Temporal embedding for each node.
            Shape: [N, output_dim]
        """

        if H.dim() != 3:
            raise ValueError(
                f"H must have shape [history_len, N, input_dim], got {tuple(H.shape)}"
            )

        history_len, N, F = H.shape

        if F != self.input_dim:
            raise ValueError(
                f"Expected input_dim={self.input_dim}, but got last dimension {F}."
            )

        # Rearrange so each node becomes one sequence sample:
        # [history_len, N, input_dim] -> [N, history_len, input_dim]
        H_nodes = H.permute(1, 0, 2).contiguous()

        # LSTM output:
        #   lstm_out : [N, history_len, output_dim]
        #   h_n      : [num_layers * num_directions, N, lstm_hidden_dim]
        lstm_out, (h_n, c_n) = self.lstm(H_nodes)

        if self.use_last_timestep:
            # Use the output at the final history step
            node_temporal = lstm_out[:, -1, :]   # [N, output_dim]
        else:
            # Alternative: use the final hidden state
            if self.bidirectional:
                # last layer forward + last layer backward
                h_forward = h_n[-2]   # [N, lstm_hidden_dim]
                h_backward = h_n[-1]  # [N, lstm_hidden_dim]
                node_temporal = torch.cat([h_forward, h_backward], dim=-1)
            else:
                node_temporal = h_n[-1]  # [N, lstm_hidden_dim]

        node_temporal = self.norm(node_temporal)
        node_temporal = self.dropout_layer(node_temporal)

        return node_temporal
    

    # -------------------------------------------------------------
# Quick shape test for NodeTemporalLSTM
# -------------------------------------------------------------
sample = dataset[0]

graph = sample["graph"] if isinstance(sample, dict) else sample[0]
x_seq = sample["x_seq"] if isinstance(sample, dict) else sample[1]
edge_seq = sample["edge_seq"] if isinstance(sample, dict) else sample[2]

encoder = MooringGATEncoder(
    n_static_node_features=13,
    n_dynamic_node_features=10,
    n_static_edge_features=8,
    n_dynamic_edge_features=4,
    gat_hidden_dim=64,
    gat_out_dim=64,
    num_heads=4,
    dropout=0.1,
)

# Build spatial embeddings over the whole history window
h_list = []
with torch.no_grad():
    for t in range(x_seq.shape[0]):
        x_t = x_seq[t]         # [N, 10]
        edge_t = edge_seq[t]   # [E, 4]
        h_t = encoder(graph, x_t, edge_t)   # [N, 64]
        h_list.append(h_t)

H = torch.stack(h_list, dim=0)   # [history_len, N, 64]

temporal_block = NodeTemporalLSTM(
    input_dim=64,
    lstm_hidden_dim=128,
    num_lstm_layers=1,
    dropout=0.1,
    bidirectional=False,
    use_layernorm=True,
    use_last_timestep=True,
)

with torch.no_grad():
    node_temporal = temporal_block(H)

print("H shape               :", H.shape)               # [history_len, N, 64]
print("node_temporal shape   :", node_temporal.shape)   # [N, 128]

H shape               : torch.Size([10, 7, 64])
node_temporal shape   : torch.Size([7, 128])


In [6]:
import torch
import torch.nn as nn


class MooringGATLSTM(nn.Module):
    """
    Full spatiotemporal model for the mooring-line problem.

    Pipeline:
        1) For each history time step t:
              - combine static + dynamic graph information
              - run MooringGATEncoder
              - get node embeddings h_t

        2) Stack all h_t over the history window:
              H = [history_len, N, gat_out_dim]

        3) Run NodeTemporalLSTM on H:
              node_temporal = [N, temporal_dim]

        4) Predict all future steps directly for each node:
              y_hat = [future_len, N, output_dim]

    Default output_dim=3 corresponds to:
        [u_x, u_z, tension]
    """

    def __init__(
        self,
        # ----- graph feature sizes -----
        n_static_node_features: int = 13,
        n_dynamic_node_features: int = 10,
        n_static_edge_features: int = 8,
        n_dynamic_edge_features: int = 4,

        # ----- spatial encoder -----
        gat_hidden_dim: int = 64,
        gat_out_dim: int = 64,
        num_heads: int = 4,
        gat_dropout: float = 0.1,
        gat_use_layernorm: bool = True,
        add_residual_projection: bool = True,

        # ----- temporal encoder -----
        lstm_hidden_dim: int = 128,
        num_lstm_layers: int = 1,
        lstm_dropout: float = 0.1,
        bidirectional: bool = False,
        lstm_use_layernorm: bool = True,
        use_last_timestep: bool = True,

        # ----- prediction head -----
        future_len: int = 1,
        output_dim: int = 3,
        head_hidden_dim: int = 128,
        head_dropout: float = 0.1,
    ):
        super().__init__()

        self.future_len = future_len
        self.output_dim = output_dim

        # -------------------------------------------------------------
        # Spatial encoder: one time step -> node embeddings
        # -------------------------------------------------------------
        self.spatial_encoder = MooringGATEncoder(
            n_static_node_features=n_static_node_features,
            n_dynamic_node_features=n_dynamic_node_features,
            n_static_edge_features=n_static_edge_features,
            n_dynamic_edge_features=n_dynamic_edge_features,
            gat_hidden_dim=gat_hidden_dim,
            gat_out_dim=gat_out_dim,
            num_heads=num_heads,
            dropout=gat_dropout,
            use_layernorm=gat_use_layernorm,
            add_residual_projection=add_residual_projection,
        )

        # -------------------------------------------------------------
        # Temporal encoder: sequence of node embeddings -> one temporal
        # embedding per node
        # -------------------------------------------------------------
        self.temporal_encoder = NodeTemporalLSTM(
            input_dim=gat_out_dim,
            lstm_hidden_dim=lstm_hidden_dim,
            num_lstm_layers=num_lstm_layers,
            dropout=lstm_dropout,
            bidirectional=bidirectional,
            use_layernorm=lstm_use_layernorm,
            use_last_timestep=use_last_timestep,
        )

        temporal_out_dim = self.temporal_encoder.output_dim

        # -------------------------------------------------------------
        # Prediction head:
        # [N, temporal_out_dim] -> [N, future_len * output_dim]
        # then reshape to [future_len, N, output_dim]
        # -------------------------------------------------------------
        self.prediction_head = nn.Sequential(
            nn.Linear(temporal_out_dim, head_hidden_dim),
            nn.ELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden_dim, future_len * output_dim),
        )

    def forward(self, graph, x_seq: torch.Tensor, edge_seq: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        graph : torch_geometric.data.Data
            Static graph object.
        x_seq : torch.Tensor
            Dynamic node features over the history window.
            Expected shape: [history_len, N, n_dynamic_node_features]
        edge_seq : torch.Tensor
            Dynamic edge features over the history window.
            Expected shape: [history_len, E, n_dynamic_edge_features]

        Returns
        -------
        y_hat : torch.Tensor
            Predicted future targets.
            Shape: [future_len, N, output_dim]
        """

        # -------------------------------------------------------------
        # Basic shape checks
        # -------------------------------------------------------------
        if x_seq.dim() != 3:
            raise ValueError(
                f"x_seq must have shape [history_len, N, F_dyn_node], got {tuple(x_seq.shape)}"
            )

        if edge_seq.dim() != 3:
            raise ValueError(
                f"edge_seq must have shape [history_len, E, F_dyn_edge], got {tuple(edge_seq.shape)}"
            )

        history_len_x, N_x, _ = x_seq.shape
        history_len_e, E_x, _ = edge_seq.shape

        if history_len_x != history_len_e:
            raise ValueError(
                f"History length mismatch: x_seq has {history_len_x}, edge_seq has {history_len_e}."
            )

        if graph.x.size(0) != N_x:
            raise ValueError(
                f"Node count mismatch: graph.x has {graph.x.size(0)} nodes, but x_seq has {N_x}."
            )

        if graph.edge_attr.size(0) != E_x:
            raise ValueError(
                f"Edge count mismatch: graph.edge_attr has {graph.edge_attr.size(0)} edges, "
                f"but edge_seq has {E_x}."
            )

        # -------------------------------------------------------------
        # Spatial encoding over all history steps
        # -------------------------------------------------------------
        h_list = []

        for t in range(history_len_x):
            x_t = x_seq[t]         # [N, F_dyn_node]
            edge_t = edge_seq[t]   # [E, F_dyn_edge]

            h_t = self.spatial_encoder(graph, x_t, edge_t)   # [N, gat_out_dim]
            h_list.append(h_t)

        # Stack over time:
        # H = [history_len, N, gat_out_dim]
        H = torch.stack(h_list, dim=0)

        # -------------------------------------------------------------
        # Temporal encoding
        # -------------------------------------------------------------
        node_temporal = self.temporal_encoder(H)   # [N, temporal_out_dim]

        # -------------------------------------------------------------
        # Predict all future steps directly
        # -------------------------------------------------------------
        y_hat_flat = self.prediction_head(node_temporal)  # [N, future_len * output_dim]

        # reshape to [N, future_len, output_dim]
        y_hat = y_hat_flat.view(N_x, self.future_len, self.output_dim)

        # permute to match dataset target convention: [future_len, N, output_dim]
        y_hat = y_hat.permute(1, 0, 2).contiguous()

        return y_hat
    
# -------------------------------------------------------------
# Quick shape test for the full MooringGATLSTM
# -------------------------------------------------------------
sample = dataset[0]

graph = sample["graph"] if isinstance(sample, dict) else sample[0]
x_seq = sample["x_seq"] if isinstance(sample, dict) else sample[1]
edge_seq = sample["edge_seq"] if isinstance(sample, dict) else sample[2]
y_seq = sample["y_seq"] if isinstance(sample, dict) else sample[3]

print("graph.x shape       :", graph.x.shape)
print("graph.edge_attr     :", graph.edge_attr.shape)
print("x_seq shape         :", x_seq.shape)
print("edge_seq shape      :", edge_seq.shape)
print("y_seq shape         :", y_seq.shape)

future_len = y_seq.shape[0]
output_dim = y_seq.shape[-1]

model = MooringGATLSTM(
    n_static_node_features=13,
    n_dynamic_node_features=10,
    n_static_edge_features=8,
    n_dynamic_edge_features=4,
    gat_hidden_dim=64,
    gat_out_dim=64,
    num_heads=4,
    gat_dropout=0.1,
    gat_use_layernorm=True,
    add_residual_projection=True,
    lstm_hidden_dim=128,
    num_lstm_layers=1,
    lstm_dropout=0.1,
    bidirectional=False,
    lstm_use_layernorm=True,
    use_last_timestep=True,
    future_len=future_len,
    output_dim=output_dim,
    head_hidden_dim=128,
    head_dropout=0.1,
)

with torch.no_grad():
    y_hat = model(graph, x_seq, edge_seq)

print("y_hat shape         :", y_hat.shape)   # expected [future_len, N, output_dim]
print("target shape        :", y_seq.shape)

# -------------------------------------------------------------
# Optional loss test
# -------------------------------------------------------------
criterion = nn.MSELoss()

with torch.no_grad():
    y_hat = model(graph, x_seq, edge_seq)
    loss = criterion(y_hat, y_seq)

print("test loss:", loss.item())

graph.x shape       : torch.Size([7, 13])
graph.edge_attr     : torch.Size([12, 8])
x_seq shape         : torch.Size([10, 7, 10])
edge_seq shape      : torch.Size([10, 12, 4])
y_seq shape         : torch.Size([1, 7, 3])
y_hat shape         : torch.Size([1, 7, 3])
target shape        : torch.Size([1, 7, 3])
test loss: 342883.15625


In [ ]:
# -------------------------------------------------------------
# imports and training config
# -------------------------------------------------------------

import os
import math
import copy
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

@dataclass
class TrainingConfig:
    seed: int = 42

    # split
    train_lc_ids: Tuple[int, ...] = tuple(range(1, 12))   # 1..11
    test_extra_lc_ids: Tuple[int, ...] = tuple(range(12, 21))  # 12..20
    test_time_fraction: float = 0.30
    train_fraction_within_dev_blocks: float = 0.70
    raw_block_len: int = 64   # can be tuned later

    # optimization
    num_epochs: int = 100
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 1
    grad_clip_max_norm: float = 1.0

    # scheduler / stopping
    use_scheduler: bool = True
    scheduler_factor: float = 0.5
    scheduler_patience: int = 5
    early_stopping_patience: int = 15
    min_delta: float = 1e-5

    # checkpoint
    checkpoint_dir: str = "./checkpoints_gat_lstm"

    # normalization
    eps: float = 1e-8

    # feature indices
    node_continuous_idx: Tuple[int, ...] = (0, 1, 2, 3, 4, 5, 6, 8, 9)
    node_binary_idx: Tuple[int, ...] = (7,)

    edge_continuous_idx: Tuple[int, ...] = (0, 1, 2)
    edge_binary_idx: Tuple[int, ...] = (3,)

    target_continuous_idx: Tuple[int, ...] = (0, 1, 2)


    def set_seed(seed: int = 42):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


In [ ]:
# -------------------------------------------------------------
# split utilities
# -------------------------------------------------------------

def get_window_span(dataset: MooringSequenceDataset) -> int:
    """
    Full raw-time footprint of one sample:
    [start_idx, ..., start_idx + history_len + future_len - 1]
    """
    return dataset.history_len + dataset.future_len

def valid_window_starts_inside_raw_interval(
    dataset: MooringSequenceDataset,
    raw_start: int,
    raw_end_exclusive: int,
) -> List[int]:
    """
    Return window start indices whose full input+target footprint lies
    entirely inside [raw_start, raw_end_exclusive).

    Window raw-time footprint:
        [s, s + history_len + future_len)

    because:
        input  uses [s, s+history_len)
        target uses [s+history_len, s+history_len+future_len)
    """
    span = dataset.history_len + dataset.future_len
    starts = []

    for s in range(len(dataset)):
        used_start = s
        used_end_exclusive = s + span
        if used_start >= raw_start and used_end_exclusive <= raw_end_exclusive:
            starts.append(s)

    return starts

def chunk_raw_time_range(
    raw_start: int,
    raw_end_exclusive: int,
    block_len: int,
) -> List[Tuple[int, int]]:
    """
    Split a raw-time interval into contiguous blocks:
        [(b0_start, b0_end), (b1_start, b1_end), ...]
    """
    blocks = []
    cur = raw_start
    while cur < raw_end_exclusive:
        nxt = min(cur + block_len, raw_end_exclusive)
        blocks.append((cur, nxt))
        cur = nxt
    return blocks

def build_leakage_aware_splits(
    load_case_datasets: Dict[int, MooringSequenceDataset],
    cfg: TrainingConfig,
) -> Dict[str, List[Tuple[int, int]]]:
    """
    Returns dict:
        {
            "train": [(lc_id, window_idx), ...],
            "val":   [(lc_id, window_idx), ...],
            "test":  [(lc_id, window_idx), ...],
        }

    Interpretation used:
    - LC 1..11:
        * last 30% raw time -> test
        * first 70% raw time -> development region
        * development region split randomly by non-overlapping raw-time blocks
          into train/val
    - LC 12..20:
        * all windows -> test
    """
    rng = random.Random(cfg.seed)

    split = {"train": [], "val": [], "test": []}

    for lc_id, ds in load_case_datasets.items():
        num_steps = ds.num_steps

        if lc_id in cfg.train_lc_ids:
            test_raw_start = int(math.floor((1.0 - cfg.test_time_fraction) * num_steps))
            dev_raw_start = 0
            dev_raw_end = test_raw_start

            # test windows: full footprint entirely in last 30% raw time
            test_window_ids = valid_window_starts_inside_raw_interval(
                ds,
                raw_start=test_raw_start,
                raw_end_exclusive=num_steps,
            )
            split["test"].extend((lc_id, w) for w in test_window_ids)

            # dev blocks: non-overlapping raw-time blocks in first 70%
            blocks = chunk_raw_time_range(
                raw_start=dev_raw_start,
                raw_end_exclusive=dev_raw_end,
                block_len=cfg.raw_block_len,
            )

            # Keep only blocks that can host at least one full window
            valid_blocks = []
            for b_start, b_end in blocks:
                block_window_ids = valid_window_starts_inside_raw_interval(
                    ds,
                    raw_start=b_start,
                    raw_end_exclusive=b_end,
                )
                if len(block_window_ids) > 0:
                    valid_blocks.append((b_start, b_end, block_window_ids))

            rng.shuffle(valid_blocks)

            n_train_blocks = int(math.floor(cfg.train_fraction_within_dev_blocks * len(valid_blocks)))
            train_blocks = valid_blocks[:n_train_blocks]
            val_blocks = valid_blocks[n_train_blocks:]

            for _, _, w_ids in train_blocks:
                split["train"].extend((lc_id, w) for w in w_ids)

            for _, _, w_ids in val_blocks:
                split["val"].extend((lc_id, w) for w in w_ids)

        elif lc_id in cfg.test_extra_lc_ids:
            split["test"].extend((lc_id, w) for w in range(len(ds)))

        else:
            raise ValueError(f"Load case {lc_id} is not assigned to any split rule.")

    return split

def print_split_summary(
    load_case_datasets: Dict[int, MooringSequenceDataset],
    split_indices: Dict[str, List[Tuple[int, int]]],
):
    print("----- SPLIT SUMMARY -----")
    for split_name, pairs in split_indices.items():
        counts_per_lc = {}
        for lc_id, w_idx in pairs:
            counts_per_lc[lc_id] = counts_per_lc.get(lc_id, 0) + 1

        total = len(pairs)
        print(f"\n{split_name.upper()} total windows: {total}")
        for lc_id in sorted(counts_per_lc):
            print(f"  LC {lc_id:02d}: {counts_per_lc[lc_id]}")

In [ ]:
# -------------------------------------------------------------
# subset dataset
# -------------------------------------------------------------

class MultiLoadCaseWindowSubset(Dataset):
    """
    Thin wrapper around multiple MooringSequenceDataset objects.

    Each item is still the same dictionary structure as before, plus lc_id.
    """
    def __init__(
        self,
        load_case_datasets: Dict[int, MooringSequenceDataset],
        index_pairs: List[Tuple[int, int]],
    ):
        self.load_case_datasets = load_case_datasets
        self.index_pairs = index_pairs

    def __len__(self):
        return len(self.index_pairs)

    def __getitem__(self, idx):
        lc_id, window_idx = self.index_pairs[idx]
        sample = self.load_case_datasets[lc_id][window_idx]

        out = {
            "graph": sample["graph"],
            "x_seq": sample["x_seq"],
            "edge_seq": sample["edge_seq"],
            "y_seq": sample["y_seq"],
            "start_idx": sample["start_idx"],
            "lc_id": lc_id,
        }
        return out
    
def single_item_collate(batch):
    """
    Current model is single-sample, not batched.
    So DataLoader must use batch_size=1 and this collate function.
    """
    if len(batch) != 1:
        raise ValueError("single_item_collate expects batch_size=1.")
    return batch[0]

In [ ]:
# -------------------------------------------------------------
# normalization utilities
# -------------------------------------------------------------

class FeatureStandardizer:
    """
    Per-feature z-score standardization for selected feature columns.

    Works with tensors shaped:
    - node inputs:   [history_len, N, F]
    - edge inputs:   [history_len, E, F]
    - targets:       [future_len, N, F]
    """
    def __init__(self, feature_idx: Tuple[int, ...], eps: float = 1e-8):
        self.feature_idx = list(feature_idx)
        self.eps = eps
        self.mean = None
        self.std = None

    def fit_from_tensor_list(self, tensor_list: List[torch.Tensor]):
        """
        Aggregate across all dimensions except the last feature dimension.
        """
        selected = []
        for x in tensor_list:
            xs = x[..., self.feature_idx].reshape(-1, len(self.feature_idx))
            selected.append(xs)

        big = torch.cat(selected, dim=0)
        self.mean = big.mean(dim=0)
        self.std = big.std(dim=0, unbiased=False).clamp_min(self.eps)

    def transform(self, x: torch.Tensor) -> torch.Tensor:
        x = x.clone()
        x_sel = x[..., self.feature_idx]
        x[..., self.feature_idx] = (x_sel - self.mean.to(x.device)) / self.std.to(x.device)
        return x

    def inverse_transform(self, x: torch.Tensor) -> torch.Tensor:
        x = x.clone()
        x_sel = x[..., self.feature_idx]
        x[..., self.feature_idx] = x_sel * self.std.to(x.device) + self.mean.to(x.device)
        return x
    
def fit_normalizers_from_train_subset(
    train_subset: MultiLoadCaseWindowSubset,
    cfg: TrainingConfig,
):
    node_norm = FeatureStandardizer(cfg.node_continuous_idx, eps=cfg.eps)
    edge_norm = FeatureStandardizer(cfg.edge_continuous_idx, eps=cfg.eps)
    target_norm = FeatureStandardizer(cfg.target_continuous_idx, eps=cfg.eps)

    node_tensors = []
    edge_tensors = []
    target_tensors = []

    for i in range(len(train_subset)):
        sample = train_subset[i]
        node_tensors.append(sample["x_seq"])
        edge_tensors.append(sample["edge_seq"])
        target_tensors.append(sample["y_seq"])

    node_norm.fit_from_tensor_list(node_tensors)
    edge_norm.fit_from_tensor_list(edge_tensors)
    target_norm.fit_from_tensor_list(target_tensors)

    return node_norm, edge_norm, target_norm

class NormalizedSubset(Dataset):
    """
    Applies train-fitted normalization on the fly.
    """
    def __init__(
        self,
        base_subset: MultiLoadCaseWindowSubset,
        node_norm: FeatureStandardizer,
        edge_norm: FeatureStandardizer,
        target_norm: FeatureStandardizer,
    ):
        self.base_subset = base_subset
        self.node_norm = node_norm
        self.edge_norm = edge_norm
        self.target_norm = target_norm

    def __len__(self):
        return len(self.base_subset)

    def __getitem__(self, idx):
        sample = self.base_subset[idx]

        return {
            "graph": sample["graph"],
            "x_seq": self.node_norm.transform(sample["x_seq"]),
            "edge_seq": self.edge_norm.transform(sample["edge_seq"]),
            "y_seq": self.target_norm.transform(sample["y_seq"]),
            "start_idx": sample["start_idx"],
            "lc_id": sample["lc_id"],
        }

In [ ]:
# -------------------------------------------------------------
# dataloader builder
# -------------------------------------------------------------

def build_dataloaders(
    load_case_datasets: Dict[int, MooringSequenceDataset],
    cfg: TrainingConfig,
):
    split_indices = build_leakage_aware_splits(load_case_datasets, cfg)
    print_split_summary(load_case_datasets, split_indices)

    train_raw = MultiLoadCaseWindowSubset(load_case_datasets, split_indices["train"])
    val_raw = MultiLoadCaseWindowSubset(load_case_datasets, split_indices["val"])
    test_raw = MultiLoadCaseWindowSubset(load_case_datasets, split_indices["test"])

    node_norm, edge_norm, target_norm = fit_normalizers_from_train_subset(train_raw, cfg)

    train_ds = NormalizedSubset(train_raw, node_norm, edge_norm, target_norm)
    val_ds = NormalizedSubset(val_raw, node_norm, edge_norm, target_norm)
    test_ds = NormalizedSubset(test_raw, node_norm, edge_norm, target_norm)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=single_item_collate,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        collate_fn=single_item_collate,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=1,
        shuffle=False,
        collate_fn=single_item_collate,
    )

    norms = {
        "node_norm": node_norm,
        "edge_norm": edge_norm,
        "target_norm": target_norm,
    }

    return train_loader, val_loader, test_loader, norms, split_indices

In [ ]:
# -------------------------------------------------------------
# loss and metrics
# -------------------------------------------------------------

class SequenceSmoothL1Loss(nn.Module):
    def __init__(self, beta: float = 1.0):
        super().__init__()
        self.loss_fn = nn.SmoothL1Loss(beta=beta)

    def forward(self, y_hat, y_true):
        return self.loss_fn(y_hat, y_true)
    
@torch.no_grad()
def compute_physical_mae_per_target(
    y_hat_norm: torch.Tensor,
    y_true_norm: torch.Tensor,
    target_norm: FeatureStandardizer,
    target_names: List[str],
):
    """
    Inputs are normalized tensors with shape [future_len, N, output_dim].
    Returns dict of physical-unit MAE per target.
    """
    y_hat_phys = target_norm.inverse_transform(y_hat_norm.detach().cpu())
    y_true_phys = target_norm.inverse_transform(y_true_norm.detach().cpu())

    mae = (y_hat_phys - y_true_phys).abs().mean(dim=(0, 1))
    return {name: float(mae[i].item()) for i, name in enumerate(target_names)}

In [ ]:
# -------------------------------------------------------------
# model/optimizer setup
# -------------------------------------------------------------

def build_model_from_dataset_example(
    example_dataset: MooringSequenceDataset,
    device: torch.device,
):
    model = MooringGATLSTM(
        n_static_node_features=example_dataset.graph_data.x.shape[1],
        n_dynamic_node_features=example_dataset.dynamic_features.shape[-1],
        n_static_edge_features=example_dataset.graph_data.edge_attr.shape[1],
        n_dynamic_edge_features=example_dataset.dynamic_edge_features.shape[-1],
        future_len=example_dataset.future_len,
        output_dim=example_dataset.targets.shape[-1],

        # baseline hyperparameters
        gat_hidden_dim=64,
        gat_out_dim=64,
        num_heads=4,
        gat_dropout=0.1,
        gat_use_layernorm=True,
        add_residual_projection=True,

        lstm_hidden_dim=128,
        num_lstm_layers=1,
        lstm_dropout=0.1,
        bidirectional=False,
        lstm_use_layernorm=True,
        use_last_timestep=True,

        head_hidden_dim=128,
        head_dropout=0.1,
    ).to(device)

    return model

def build_optimizer_and_scheduler(
    model: nn.Module,
    cfg: TrainingConfig,
):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    scheduler = None
    if cfg.use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=cfg.scheduler_factor,
            patience=cfg.scheduler_patience,
        )

    return optimizer, scheduler

In [ ]:
# -------------------------------------------------------------
# training and validation loops
# -------------------------------------------------------------


def move_sample_to_device(sample, device):
    return {
        "graph": sample["graph"].to(device),
        "x_seq": sample["x_seq"].to(device),
        "edge_seq": sample["edge_seq"].to(device),
        "y_seq": sample["y_seq"].to(device),
        "start_idx": sample["start_idx"],
        "lc_id": sample["lc_id"],
    }

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    cfg: TrainingConfig,
):
    model.train()

    running_loss = 0.0
    num_batches = 0

    for sample in loader:
        sample = move_sample_to_device(sample, device)

        optimizer.zero_grad()

        y_hat = model(
            sample["graph"],
            sample["x_seq"],
            sample["edge_seq"],
        )

        loss = criterion(y_hat, sample["y_seq"])
        loss.backward()

        if cfg.grad_clip_max_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_max_norm)

        optimizer.step()

        running_loss += loss.item()
        num_batches += 1

    return running_loss / max(1, num_batches)

@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
):
    model.eval()

    running_loss = 0.0
    num_batches = 0

    mae_sums = {name: 0.0 for name in target_names}

    for sample in loader:
        sample = move_sample_to_device(sample, device)

        y_hat = model(
            sample["graph"],
            sample["x_seq"],
            sample["edge_seq"],
        )

        loss = criterion(y_hat, sample["y_seq"])
        running_loss += loss.item()
        num_batches += 1

        mae_dict = compute_physical_mae_per_target(
            y_hat_norm=y_hat,
            y_true_norm=sample["y_seq"],
            target_norm=target_norm,
            target_names=target_names,
        )

        for k, v in mae_dict.items():
            mae_sums[k] += v

    avg_loss = running_loss / max(1, num_batches)
    avg_mae = {k: v / max(1, num_batches) for k, v in mae_sums.items()}

    return avg_loss, avg_mae

In [ ]:
# -------------------------------------------------------------
# full training runner with checkpointing and early stopping
# -------------------------------------------------------------

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion,
    device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
    cfg: TrainingConfig,
):
    checkpoint_dir = Path(cfg.checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = checkpoint_dir / "best_mooring_gat_lstm.pt"

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_mae": [],
    }

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, cfg.num_epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            cfg=cfg,
        )

        val_loss, val_mae = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            target_norm=target_norm,
            target_names=target_names,
        )

        if scheduler is not None:
            scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_mae"].append(val_mae)

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.6f} | "
            f"val_loss={val_loss:.6f} | "
            + " | ".join([f"val_MAE_{k}={v:.6f}" for k, v in val_mae.items()])
        )

        improved = (best_val_loss - val_loss) > cfg.min_delta
        if improved:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, best_model_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= cfg.early_stopping_patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, str(best_model_path)

In [ ]:
# -------------------------------------------------------------
# test evaluation
# -------------------------------------------------------------


@torch.no_grad()
def test_model(
    model,
    test_loader,
    criterion,
    device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
):
    test_loss, test_mae = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
        target_norm=target_norm,
        target_names=target_names,
    )

    print("\n----- TEST RESULTS -----")
    print(f"test_loss = {test_loss:.6f}")
    for k, v in test_mae.items():
        print(f"test_MAE_{k} = {v:.6f}")

    return {
        "test_loss": test_loss,
        "test_mae": test_mae,
    }

In [ ]:
# -------------------------------------------------------------
# end-to-end training execution cell
# -------------------------------------------------------------


# -------------------------------------------------------------
# 1. USER MUST PROVIDE:
#    load_case_datasets = {lc_id: MooringSequenceDataset(...), ...}
# -------------------------------------------------------------
# Example expected keys:
# load_case_datasets.keys() == {1, 2, ..., 20}

# Safety checks
assert isinstance(load_case_datasets, dict), "load_case_datasets must be a dict."
assert len(load_case_datasets) > 0, "load_case_datasets is empty."

first_lc_id = sorted(load_case_datasets.keys())[0]
example_dataset = load_case_datasets[first_lc_id]

# Consistency checks across load cases
ref_num_nodes = example_dataset.graph_data.num_nodes_total
ref_node_feat = example_dataset.graph_data.x.shape[1]
ref_edge_feat = example_dataset.graph_data.edge_attr.shape[1]
ref_dyn_node_feat = example_dataset.dynamic_features.shape[-1]
ref_dyn_edge_feat = example_dataset.dynamic_edge_features.shape[-1]
ref_target_dim = example_dataset.targets.shape[-1]
ref_history_len = example_dataset.history_len
ref_future_len = example_dataset.future_len

for lc_id, ds in load_case_datasets.items():
    assert ds.graph_data.num_nodes_total == ref_num_nodes, f"LC {lc_id}: num_nodes mismatch"
    assert ds.graph_data.x.shape[1] == ref_node_feat, f"LC {lc_id}: static node feature mismatch"
    assert ds.graph_data.edge_attr.shape[1] == ref_edge_feat, f"LC {lc_id}: static edge feature mismatch"
    assert ds.dynamic_features.shape[-1] == ref_dyn_node_feat, f"LC {lc_id}: dynamic node feature mismatch"
    assert ds.dynamic_edge_features.shape[-1] == ref_dyn_edge_feat, f"LC {lc_id}: dynamic edge feature mismatch"
    assert ds.targets.shape[-1] == ref_target_dim, f"LC {lc_id}: target dim mismatch"
    assert ds.history_len == ref_history_len, f"LC {lc_id}: history_len mismatch"
    assert ds.future_len == ref_future_len, f"LC {lc_id}: future_len mismatch"

print("All load cases passed consistency checks.")


cfg = TrainingConfig()
set_seed(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_loader, val_loader, test_loader, norms, split_indices = build_dataloaders(
    load_case_datasets=load_case_datasets,
    cfg=cfg,
)

model = build_model_from_dataset_example(
    example_dataset=example_dataset,
    device=device,
)

criterion = SequenceSmoothL1Loss(beta=1.0)
optimizer, scheduler = build_optimizer_and_scheduler(model, cfg)

model, history, best_model_path = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device,
    target_norm=norms["target_norm"],
    target_names=example_dataset.target_feature_names,
    cfg=cfg,
)

print("\nBest model saved to:", best_model_path)

test_results = test_model(
    model=model,
    test_loader=test_loader,
    criterion=criterion,
    device=device,
    target_norm=norms["target_norm"],
    target_names=example_dataset.target_feature_names,
)